# Segment B — Build the Tool-Use Loop by Hand
### Module 1: Claude API & Ecosystem Foundations

**Goal:** define two real tools (Wikipedia search, arXiv search), write the full
tool-use loop ourselves — no framework — and watch Claude choose which tool to use
based on the question.

This is the most important notebook of the day. Everything in Modules 2–4 is a
more convenient way of doing exactly what we're about to do by hand.

## 1. Setup

In [ ]:
import os
import json
import urllib.request
import urllib.parse
from anthropic import Anthropic

os.environ["ANTHROPIC_API_KEY"] = "sk-..."

client = Anthropic()
MODEL = "claude-sonnet-4-6"


## 2. Define the actual tool functions

These are plain Python functions. Nothing Claude-specific about them yet — they
just hit a public API and return text. We'll use:

- **Wikipedia's Search API** — no key required
- **arXiv's API** — no key required

Both return XML/JSON we need to parse into something readable.

In [10]:
def search_wikipedia(query: str, limit: int = 3) -> str:
    """Search Wikipedia and return short summaries of the top results."""
    url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit,
    })
    req = urllib.request.Request(url, headers={"User-Agent": "Module1-ResearchCompanion/1.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())

    results = data.get("query", {}).get("search", [])
    if not results:
        return f"No Wikipedia results found for '{query}'."

    lines = []
    for r in results:
        title = r["title"]
        # Wikipedia search snippets include HTML <span> tags around matches; strip them
        snippet = r["snippet"].replace('<span class="searchmatch">', "").replace("</span>", "")
        lines.append(f"- {title}: {snippet}")
    return "\n".join(lines)


# Quick manual test — run this once to confirm the function itself works
# before handing it to Claude.
print(search_wikipedia("agentic AI"))

- AI agent: artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents that can pursue goals, use
- Agentic commerce: Agentic commerce (also referred to as agent-based commerce) describes an emerging form of e-commerce in which autonomous artificial intelligence (AI) agents
- Codex (AI agent): tool as gateway to AI agents for business&quot;. Fortune. Retrieved March 12, 2026. Knight, Will (May 16, 2025). &quot;OpenAI Launches an Agentic, Web-Based Coding


In [11]:
def search_arxiv(query: str, max_results: int = 3) -> str:
    """Search arXiv and return titles, authors, and abstract snippets."""
    url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
    })
    with urllib.request.urlopen(url, timeout=10) as resp:
        raw = resp.read()

    # arXiv returns Atom XML, not JSON — parse with the standard library.
    import xml.etree.ElementTree as ET
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(raw)
    entries = root.findall("atom:entry", ns)

    if not entries:
        return f"No arXiv results found for '{query}'."

    lines = []
    for e in entries:
        title = e.find("atom:title", ns).text.strip().replace("\n", " ")
        authors = [a.find("atom:name", ns).text for a in e.findall("atom:author", ns)]
        summary = e.find("atom:summary", ns).text.strip().replace("\n", " ")[:200]
        lines.append(f"- {title} ({', '.join(authors)}): {summary}...")
    return "\n".join(lines)


# Manual test
print(search_arxiv("agentic retrieval"))

- Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval and Tool Use (Francisco Javier Arceo, Varsha Prasad Narsing): Retrieval-Augmented Generation (RAG) and agentic AI systems are increasingly prevalent in enterprise AI deployments. However, real enterprise environments introduce challenges largely absent from acad...
- LLandMark: A Multi-Agent Framework for Landmark-Aware Multimodal Interactive Video Retrieval (Minh-Chi Phung, Thien-Bao Le, Cam-Tu Tran-Thi, Thu-Dieu Nguyen-Thi, Vu-Hung Dao): The increasing diversity and scale of video data demand retrieval systems capable of multimodal understanding, adaptive reasoning, and domain-specific knowledge integration. This paper presents LLandM...
- Overcoming low-utility facets for complex answer retrieval (Sean MacAvaney, Andrew Yates, Arman Cohan, Luca Soldaini, Kai Hui, Nazli Goharian, Ophir Frieder): Many questions cannot be answered simply; their answers must include numerous nuanced details and additional context. Co

## 3. Describe these tools to Claude

A Python function isn't a "tool" to Claude until we describe it in the shape the
API expects: a `name`, a `description` (this is what Claude uses to *decide* whether
to call it — write it like documentation, not a label), and an `input_schema`
(JSON Schema) describing the arguments.

In [5]:
tools = [
    {
        "name": "search_wikipedia",
        "description": (
            "Search Wikipedia for general background on a topic, person, place, "
            "or concept. Good for established facts and explanations. Not useful "
            "for very recent events or cutting-edge research."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query, e.g. a topic or concept name."
                }
            },
            "required": ["query"],
        },
    },
    {
        "name": "search_arxiv",
        "description": (
            "Search arXiv for academic papers. Good for cutting-edge research, "
            "technical methods, and citations to specific papers. Not useful for "
            "general background or non-technical topics."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query — topic, method name, or keywords."
                }
            },
            "required": ["query"],
        },
    },
]

## 4. One tool call, inspected by hand (before we automate it)

Let's send a question and a tool definition, and look at exactly what comes back —
**before** wrapping any of this in a loop. This is the slide from lecture made real.

In [12]:
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=[
        {"role": "user", "content": "What's the latest research on agentic retrieval methods?"}
    ],
)

print(response.model_dump_json(indent=2))

{
  "id": "msg_01VXWcrWpafxJmbqVJmgwG9J",
  "container": null,
  "content": [
    {
      "id": "toolu_01DMpPKMnigrSjqRGCVCChjA",
      "caller": {
        "type": "direct"
      },
      "input": {
        "query": "agentic retrieval methods"
      },
      "name": "search_arxiv",
      "type": "tool_use"
    }
  ],
  "model": "claude-sonnet-4-6",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "tool_use",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 717,
    "output_tokens": 60,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


Look at `response.stop_reason` — it should be `"tool_use"`. And `response.content`
should contain a block with `type == "tool_use"`, a `name` matching one of our tools,
and an `input` dict with the arguments Claude generated.

This is exactly the round-trip from the lecture diagram. Claude hasn't *done*
anything yet — it's asked us to.

In [13]:
for block in response.content:
    print(block.type)
    if block.type == "tool_use":
        print("  tool name:", block.name)
        print("  tool id:  ", block.id)
        print("  input:    ", block.input)

tool_use
  tool name: search_arxiv
  tool id:   toolu_01DMpPKMnigrSjqRGCVCChjA
  input:     {'query': 'agentic retrieval methods'}


## 5. Execute the tool call and send the result back

Now we close the loop manually, once, so the mechanics are fully visible:
1. Map the tool name Claude asked for to our actual Python function
2. Call it with the arguments Claude provided
3. Package the result as a `tool_result` block
4. Send it back as a new message with `role: "user"`

In [14]:
TOOL_FUNCTIONS = {
    "search_wikipedia": search_wikipedia,
    "search_arxiv": search_arxiv,
}

# Grab the tool_use block from the previous response
tool_use_block = next(b for b in response.content if b.type == "tool_use")

# Execute the actual function
function_to_call = TOOL_FUNCTIONS[tool_use_block.name]
result_text = function_to_call(**tool_use_block.input)

print(result_text)

- Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval and Tool Use (Francisco Javier Arceo, Varsha Prasad Narsing): Retrieval-Augmented Generation (RAG) and agentic AI systems are increasingly prevalent in enterprise AI deployments. However, real enterprise environments introduce challenges largely absent from acad...
- LLandMark: A Multi-Agent Framework for Landmark-Aware Multimodal Interactive Video Retrieval (Minh-Chi Phung, Thien-Bao Le, Cam-Tu Tran-Thi, Thu-Dieu Nguyen-Thi, Vu-Hung Dao): The increasing diversity and scale of video data demand retrieval systems capable of multimodal understanding, adaptive reasoning, and domain-specific knowledge integration. This paper presents LLandM...
- Overcoming low-utility facets for complex answer retrieval (Sean MacAvaney, Andrew Yates, Arman Cohan, Luca Soldaini, Kai Hui, Nazli Goharian, Ophir Frieder): Many questions cannot be answered simply; their answers must include numerous nuanced details and additional context. Co

In [15]:
# Now send that result back. Note the conversation history so far:
# 1. our original user message
# 2. Claude's tool_use response
# 3. our tool_result, sent as role="user"

messages = [
    {"role": "user", "content": "What's the latest research on agentic retrieval methods?"},
    {"role": "assistant", "content": response.content},
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": result_text,
            }
        ],
    },
]

final_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=messages,
)

for block in final_response.content:
    if block.type == "text":
        print(block.text)
print()
print("stop_reason:", final_response.stop_reason)



stop_reason: tool_use


Notice the `tool_use_id` — it has to match the `id` from Claude's original
`tool_use` block exactly. This is how Claude matches your result back to its
request, especially important if it ever requests multiple tool calls at once.

## 6. Now automate it: the actual loop

We just did the loop manually, once. Real questions might need this to repeat —
Claude might want to call a second tool after seeing the first result, or decide
it has enough information and stop. Let's write a function that keeps going until
`stop_reason` is no longer `"tool_use"`.

In [17]:
def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    Run the full tool-use loop for a single user message.
    Returns Claude's final text answer once stop_reason == "end_turn".
    """
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(max_iterations):
        response = client.messages.create(
            model=MODEL,
            max_tokens=800,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason != "tool_use":
            # Claude is done — return its final text
            return response.content[0].text

        # Otherwise, Claude wants to call one or more tools.
        # Append Claude's own turn to history first.
        messages.append({"role": "assistant", "content": response.content})

        # Execute every tool_use block in this response (there can be more than one)
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            print(f"  [iteration {iteration}] calling {block.name}({block.input})")
            function_to_call = TOOL_FUNCTIONS[block.name]
            output = function_to_call(**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": output,
            })

        messages.append({"role": "user", "content": tool_results})

    return "Reached max_iterations without a final answer — check for a loop."

## 7. Try it — and watch Claude choose

Ask something that should pull from Wikipedia, then something that should pull
from arXiv, then something ambiguous. Watch the printed tool calls to see which
one Claude picks and why — this is the first genuinely "agentic" moment of the
day: Claude is making a decision, not just following a script.

In [18]:
print(run_agent("Give me a general overview of what reinforcement learning is."))

  [iteration 0] calling search_wikipedia({'query': 'reinforcement learning'})
  [iteration 1] calling search_wikipedia({'query': 'reinforcement learning overview agent reward environment policy'})
Here is a general overview of **Reinforcement Learning (RL)**:

---

## 🤖 What is Reinforcement Learning?

**Reinforcement Learning (RL)** is a branch of **machine learning** where an intelligent **agent** learns how to make decisions by interacting with an **environment**. Rather than being explicitly told what to do, the agent learns through **trial and error**, receiving feedback in the form of **rewards** or **penalties** based on the actions it takes.

---

## 🔑 Core Concepts

| Concept | Description |
|---|---|
| **Agent** | The learner or decision-maker (e.g., a robot, a game-playing AI) |
| **Environment** | The world the agent interacts with |
| **State** | A representation of the current situation of the environment |
| **Action** | A choice the agent can make at any given state |
|

In [19]:
print(run_agent("Find a recent paper specifically about multi-agent orchestration."))

  [iteration 0] calling search_arxiv({'query': 'multi-agent orchestration'})
  [iteration 1] calling search_arxiv({'query': 'multi-agent orchestration LLM framework'})
  [iteration 2] calling search_arxiv({'query': 'orchestration of autonomous agents task delegation coordination 2024'})
  [iteration 3] calling search_arxiv({'query': 'agentic AI systems orchestrator subagent workflow 2025'})
  [iteration 4] calling search_arxiv({'query': 'multi-agent system orchestration coordination planning large language models'})
Reached max_iterations without a final answer — check for a loop.


In [ ]:
# An ambiguous one — see which tool Claude reaches for, and whether it uses both.
print(run_agent("What is Model Context Protocol, and is there academic research on it?"))

## Recap

- A "tool" is just a Python function plus a description Claude can read
- `description` is doing real work — it's how Claude decides *whether* and *which* tool to call
- The loop is: send messages → check `stop_reason` → if `tool_use`, execute + append result → repeat → return on `end_turn`
- Claude can request multiple tool calls in a single turn — our loop handles that by looping over every `tool_use` block, not just the first

Next: Segment C — turn this into a reusable, multi-turn agent and move it into a script.